# Chapter 6 · Communication Protocols Between Agents
**Mastering Agentic AI** — Manning Publications

## What this notebook covers
- **Section 6.2** — HITL gate: review queue, graduated autonomy, behavioural telemetry
- **Section 6.3** — Google A2A protocol: AgentCard, cross-framework demo, full error handling
- **Section 6.4** — IBM ACP enterprise messaging (conceptual walkthrough)
- **Section 6.5** — LLM Council three-stage deliberation with inline implementation
- **Section 6.6** — MCP + A2A two-layer composition
- **Section 6.7** — Production patterns: circuit breaker, event-driven vs synchronous

**The two-layer principle:**
- **MCP** = tool layer → data sources that respond but do not reason
- **A2A** = agent layer → peers that reason, plan, have their own tools
- Every production agent uses both simultaneously. Design them as separate concerns.

---

## 6.0 · Setup

In [ ]:
# !pip install google-adk a2a-sdk crewai uvicorn mcp httpx openai redis python-dotenv
import asyncio
import threading
import time

import uvicorn

from crewai import Agent as CrewAgent, Crew, Task
from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentSkill, TransportProtocol
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH
from google.adk.a2a.executor.a2a_agent_executor import (
    A2aAgentExecutor,
    A2aAgentExecutorConfig,
)
from google.adk.agents import Agent as AdkAgent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

print("Ready")

## 6.2 · Human-in-the-Loop Gate

HITL is load-bearing infrastructure, not a support feature. The `HITLGate` implements three lanes:

| Lane | Condition | What happens |
|------|-----------|-------------|
| **Auto-approve** | `stake=low` AND `confidence ≥ 0.85` | Proceed immediately; log for auditing |
| **Review queue** | Medium stakes | Block until human approves/rejects (5-min timeout) |
| **Escalate** | `stake=high` OR `confidence < 0.5` | Block immediately; notify human |

**Behavioural telemetry:** every outcome is logged — including the confidence score and stake level. Over time this log reveals where the agent's calibration is off and which task classes need confidence threshold adjustment.

In [ ]:
import asyncio
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum


class ReviewStatus(Enum):
    PENDING  = 'pending'
    APPROVED = 'approved'
    REJECTED = 'rejected'


@dataclass
class PendingAction:
    action_id:   str
    description: str
    payload:     dict
    confidence:  float        # 0.0–1.0
    stake_level: str          # 'low' | 'medium' | 'high'
    status:      ReviewStatus = ReviewStatus.PENDING
    created_at:  str = field(default_factory=lambda: datetime.now().isoformat())


class HITLGate:
    def __init__(self):
        self.queue = []
        self.log   = []

    async def propose(self, action: PendingAction) -> bool:
        if action.stake_level == 'low' and action.confidence >= 0.85:
            self._log(action, 'auto-approved'); return True
        if action.stake_level == 'high' or action.confidence < 0.5:
            self._log(action, 'escalated'); return False
        self.queue.append(action)
        return await self._wait_for_review(action)

    async def _wait_for_review(self, action: PendingAction) -> bool:
        for _ in range(300):
            await asyncio.sleep(1)
            if action.status == ReviewStatus.APPROVED:
                self._log(action, 'approved'); return True
            if action.status == ReviewStatus.REJECTED:
                self._log(action, 'rejected'); return False
        self._log(action, 'timed-out'); return False

    def approve(self, action_id: str):
        for a in self.queue:
            if a.action_id == action_id: a.status = ReviewStatus.APPROVED

    def reject(self, action_id: str):
        for a in self.queue:
            if a.action_id == action_id: a.status = ReviewStatus.REJECTED

    def _log(self, action: PendingAction, outcome: str):
        self.log.append({'action_id': action.action_id, 'outcome': outcome,
                         'confidence': action.confidence, 'stake': action.stake_level})


# Demo — all three lanes
async def demo_hitl():
    gate = HITLGate()
    low    = PendingAction('a1', 'Categorise ticket #4421', {}, 0.92, 'low')
    high   = PendingAction('a2', 'Wire transfer £50,000', {}, 0.88, 'high')
    medium = PendingAction('a3', 'Send refund email', {}, 0.75, 'medium')
    medium.status = ReviewStatus.REJECTED  # simulate human rejection

    for action in [low, high, medium]:
        result = await gate.propose(action)
        print(f"  {action.description[:40]:42s} → {'proceed' if result else 'blocked'}")

    print(f'\nAudit log ({len(gate.log)} entries):')
    for e in gate.log:
        print(f"  {e['action_id']} | {e['outcome']:12s} | confidence={e['confidence']} | stake={e['stake']}")

await demo_hitl()


## 6.1 · ADK Workout Agent

Define a Google ADK agent and its **AgentCard** — the A2A-standard capability descriptor that lets other agents discover what this agent can do.

In [ ]:
# ADK side: define one agent and publish it through A2A.
workout_agent = AdkAgent(
    model="gemini-2.5-pro",
    name="workout_agent",
    instruction=(
        "Create short, beginner-friendly running plans. "
        "Reply with only the plan."
    ),
)

workout_agent_card = AgentCard(
    name="Workout Agent",
    url="http://127.0.0.1:10020",
    description="Creates running plans for beginners.",
    version="1.0",
    capabilities=AgentCapabilities(streaming=False),
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id="build_running_plan",
            name="Build Running Plan",
            description="Creates a beginner-friendly 5k plan.",
            examples=["Create a 4-week 5k plan for a beginner."],
        )
    ],
)

print(f"Agent: {workout_agent_card.name}")
print(f"URL:   {workout_agent_card.url}")
print(f"Skills: {[s.name for s in workout_agent_card.skills]}")

## 6.3.2 · A2A Server Infrastructure

Wrap the ADK agent in an A2A-compliant HTTP server so any agent (regardless of framework) can discover and call it.

In [ ]:
def create_agent_a2a_server(agent, agent_card):
    runner = Runner(
        app_name=agent.name,
        agent=agent,
        artifact_service=InMemoryArtifactService(),
        session_service=InMemorySessionService(),
        memory_service=InMemoryMemoryService(),
    )

    executor = A2aAgentExecutor(
        runner=runner,
        config=A2aAgentExecutorConfig(),
    )

    handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )

    return A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=handler,
    )


async def run_workout_server():
    app = create_agent_a2a_server(workout_agent, workout_agent_card)
    config = uvicorn.Config(
        app.build(),
        host="127.0.0.1",
        port=10020,
        log_level="warning",
        loop="none",
    )
    server = uvicorn.Server(config)
    await server.serve()


def start_server_in_background():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(run_workout_server())


print("A2A server functions defined.")

## 6.3 · CrewAI Diet Coach + A2A Tool

The CrewAI Diet Coach discovers the ADK Workout Agent via its A2A `AgentCard` and calls it over HTTP.

**Imports required for the A2A client** (shown explicitly — a missing import here causes `ImportError`):
```python
from a2a.client import ClientFactory, ClientConfig, create_text_message_object
# ClientFactory              — creates a protocol-aware A2A client from an AgentCard
# ClientConfig               — holds transport configuration (timeouts, retries)
# create_text_message_object — wraps a string in the A2A message envelope
```

The `CrewA2ATool` wrapper handles the three production failure cases in order of specificity:
1. `ConnectionError` — network unreachable; agent server not running
2. `TimeoutError` — server reached but call hung without completing
3. `Exception` — any other unexpected error; logged and surfaced

In [ ]:
# Section 6.3.4 — A2A client imports (explicit — prevents ImportError)
from a2a.client import ClientFactory, ClientConfig, create_text_message_object


class CrewA2ATool:
    """
    Section 6.3.4: A2A client wrapper for the Workout Agent.

    Three failure cases handled in order of specificity:
      • ConnectionError — network unreachable; agent server not running
      • TimeoutError    — server reached but call hung without completing
      • Exception       — any other unexpected error; logged and surfaced
    """

    async def ask(self, prompt: str) -> str:
        try:
            client  = ClientFactory(ClientConfig()).create(workout_agent_card)
            message = create_text_message_object(content=prompt)
            async for response in client.send_message(message):
                task = response[0]
                return task.artifacts[0].parts[0].root.text
            return 'No response received.'
        except ConnectionError as e:
            return f'[Error] Could not reach Workout Agent: {e}'
        except TimeoutError:
            return '[Error] Workout Agent timed out — retry or escalate.'
        except Exception as e:
            return f'[Error] A2A call failed: {type(e).__name__}: {e}'


print('CrewA2ATool defined with full error handling.')
print('Handles: ConnectionError → TimeoutError → Exception (in specificity order)')


## 6.3.5 · Run the Cross-Framework Demo

Start the ADK Workout Agent as a background A2A server, then have the CrewAI Diet Coach discover and call it.

In [ ]:
threading.Thread(target=start_server_in_background, daemon=True).start()
time.sleep(2)
await crewai_agent_calls_adk()

## 6.5 · LLM Council — Three-Stage Deliberation

Instead of querying one model, the Council pattern deliberates across multiple:

| Stage | What happens | Why |
|-------|-------------|-----|
| **1. Independent generation** | Each model answers without seeing others | Prevents anchoring bias |
| **2. Anonymous cross-review** | Each model ranks shuffled answers | No favouritism by model identity |
| **3. Chairman synthesis** | One model integrates all views | Produces a single high-confidence output |

**Cost:** `(2 × len(COUNCIL_MODELS) + 1)` model calls per question.  
**Warning:** Diversity in council composition matters — different model *families*, not just different sizes of the same model. Correlated training data = correlated errors = majority vote fails.

In [ ]:
import random
from openai import OpenAI

council_client = OpenAI()
COUNCIL_MODELS = ['gpt-4o', 'gpt-4o-mini']  # add models via OpenRouter


def _ask(model: str, question: str) -> str:
    return council_client.chat.completions.create(
        model=model, max_tokens=300,
        messages=[{'role': 'user', 'content': question}]
    ).choices[0].message.content


def _review(model: str, question: str, answers: list) -> str:
    prompt = (f'Question: {question}\n\nAnonymous answers:\n'
              + '\n---\n'.join(answers)
              + '\n\nRank these by accuracy. Explain briefly.')
    return _ask(model, prompt)


def _synthesise(model: str, question: str, opinions: dict, reviews: dict) -> str:
    context = '\n'.join(f'{m}: {a}' for m, a in opinions.items())
    rev_ctx = '\n'.join(f'{m}: {r}' for m, r in reviews.items())
    prompt  = (f'Question: {question}\nAll answers:\n{context}\n'
               f'Reviews:\n{rev_ctx}\n\nSynthesise the best answer.')
    return _ask(model, prompt)


def council_query(question: str, chairman: str = 'gpt-4o') -> dict:
    """Run a full LLM Council deliberation. Returns opinions, reviews, synthesis."""
    opinions = {m: _ask(m, question) for m in COUNCIL_MODELS}
    anon = list(opinions.values()); random.shuffle(anon)
    reviews  = {m: _review(m, question, anon) for m in COUNCIL_MODELS}
    synthesis = _synthesise(chairman, question, opinions, reviews)
    return {'opinions': opinions, 'reviews': reviews, 'synthesis': synthesis}


# Demo
result = council_query('Should an AI agent summarise emails before sending them?')
print('=== Opinions ===')
for model, opinion in result['opinions'].items():
    print(f'  [{model}] {opinion[:120]}...')
print('\n=== Synthesis ===')
print(result['synthesis'][:300])


## 6.5.6 · Composing MCP and A2A in a Single Agent

MCP and A2A are not competing standards — they operate on **different planes**:

| Protocol | Layer | What it connects | Nature of downstream |
|----------|-------|-----------------|---------------------|
| **MCP** | Tool layer | Data sources, APIs, services | Responds to queries — no agency |
| **A2A** | Agent layer | Peer agents | Reasons, plans, has own tools |

A production agent sits between **both layers simultaneously**:
- It uses **MCP** to reach tools below it (nutrition DB, meal logger)
- It uses **A2A** to reach peer agents beside it (Workout Agent, future specialists)

The two protocols are **orthogonal and independent** — they compose cleanly in the same agent loop.

```
┌──────────────────────────────────────────────────┐
│               Diet Coach (ADK)                   │
│  MCP tools (own): nutrition DB, meal logger      │
│  A2A client: calls Workout Agent over the wire   │
└───────────────┬──────────────────────────────────┘
                │ A2A
┌───────────────▼──────────────────────────────────┐
│           Workout Agent (ADK) — A2A server       │
│  MCP tools (internal): fitness DB                │
│  Invisible to the Diet Coach — fully encapsulated│
└──────────────────────────────────────────────────┘
```

**Key insight:** The caller never sees the callee's internal MCP connections. Each agent manages its own tool layer privately; only its A2A interface is public.

In [ ]:
# Section 6.5.6 — Additional imports for MCP + A2A composition
import os
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from google.adk.tools.toolbox_toolset import ToolboxToolset
from mcp import StdioServerParameters

# Set NUTRITION_DB_URL in your .env to connect to a real MCP Toolbox server.
# Leave unset to use the simulated result in this notebook.
NUTRITION_DB_URL = os.getenv('NUTRITION_DB_URL', '')
print('MCP + A2A imports ready.')
print(f'NUTRITION_DB_URL: {NUTRITION_DB_URL or "(not set — will simulate)"}')


In [ ]:
# ── Step 1: Workout Agent with internal MCP tools ─────────────────────────
# The fitness DB connection is entirely internal — invisible to A2A callers.
# This is the key encapsulation principle: each agent owns its tool layer;
# only its agent interface is public.

def build_fitness_mcp_tools():
    """Connect the Workout Agent to a fitness database via MCP."""
    try:
        return McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command='npx',
                    args=['-y', 'fitness-db-mcp-server'],
                ),
                timeout=30,
            )
        )
    except Exception as e:
        print(f'[MCP] Fitness DB unavailable: {e}')
        return None


def build_mcp_equipped_workout_agent():
    """Workout Agent with MCP fitness tools encapsulated inside."""
    fitness_tools = build_fitness_mcp_tools()
    tools = [fitness_tools] if fitness_tools else []
    return AdkAgent(
        model='gemini-2.5-pro',
        name='workout_agent_mcp',
        instruction=(
            'Create beginner-friendly running plans. '
            'Use the fitness database to check exercise load norms '
            'and weekly mileage recommendations when available.'
        ),
        tools=tools,
    )


# AgentCard for the MCP-equipped Workout Agent.
# Describes the public A2A interface only — MCP is not mentioned.
workout_agent_mcp_card = AgentCard(
    name='Workout Agent (MCP-equipped)',
    url='http://127.0.0.1:10021',
    description='Creates running plans using fitness data from an MCP database.',
    version='2.0',
    capabilities=AgentCapabilities(streaming=False),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='build_running_plan_with_data',
            name='Build Data-Backed Running Plan',
            description='5k plan informed by real fitness database norms.',
            examples=['Create a 4-week 5k plan for a beginner.'],
        )
    ],
)

print(f'Agent: {workout_agent_mcp_card.name}')
print(f'URL:   {workout_agent_mcp_card.url}')
print('MCP fitness tools: encapsulated inside agent — invisible to A2A callers')


In [ ]:
# ── Step 2: Publish the MCP-equipped Workout Agent over A2A ──────────────
# Two protocols, two layers, fully decoupled.
# The agent uses MCP internally; externally it is a standard A2A server.

async def run_mcp_workout_server():
    agent = build_mcp_equipped_workout_agent()
    app = create_agent_a2a_server(agent, workout_agent_mcp_card)
    config = uvicorn.Config(
        app.build(),
        host='127.0.0.1',
        port=10021,
        log_level='warning',
        loop='none',
    )
    await uvicorn.Server(config).serve()


def start_mcp_server_in_background():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(run_mcp_workout_server())


# Start the MCP-equipped Workout Agent server
threading.Thread(target=start_mcp_server_in_background, daemon=True).start()
time.sleep(2)
print('MCP-equipped Workout Agent serving on http://127.0.0.1:10021')
print('Internal MCP fitness tools: active (if npx + server available)')
print('External A2A interface: ready for any calling agent')


In [ ]:
# ── Step 3: Diet Coach uses MCP and A2A simultaneously ───────────────────
# Same workflow — two protocols, two layers, one synthesised response.

class McpWorkoutA2ATool(CrewA2ATool):
    """A2A client pointed at the MCP-equipped Workout Agent (port 10021)."""
    async def ask(self, prompt: str) -> str:
        client = ClientFactory(ClientConfig()).create(workout_agent_mcp_card)
        message = create_text_message_object(content=prompt)
        async for response in client.send_message(message):
            task = response[0]
            return task.artifacts[0].parts[0].root.text
        return 'No response received.'


async def demonstrate_mcp_and_a2a_composition():
    print('── MCP + A2A Composition Demo ────────────────────────────────────')

    # MCP call (simulated): Diet Coach queries its own nutrition DB
    # In production: nutrition_tools = ToolboxToolset(server_url=NUTRITION_DB_URL)
    print('[MCP → Nutrition DB] Querying salmon nutritional data...')
    nutrition_result = {
        'food': 'Atlantic salmon (100g)', 'calories': 208,
        'protein': '28g', 'fat': '12g', 'carbs': '0g',
    }
    print(f'  Result: {nutrition_result}\n')

    # A2A call: Diet Coach delegates to Workout Agent over A2A
    # Inside the Workout Agent, MCP queries the fitness DB — invisible here
    print('[A2A → Workout Agent] Requesting 4-week 5k plan...')
    mcp_a2a_tool = McpWorkoutA2ATool()
    workout_plan = await mcp_a2a_tool.ask(
        'Create a beginner-friendly 4-week 5k running plan. '
        'Use fitness database norms for weekly mileage if available.'
    )
    print(f'  Workout Agent replied: {workout_plan[:150]}...\n')

    # Diet Coach synthesises both results into one response
    print('[Diet Coach] Synthesising results...')
    print(f"  '{nutrition_result['food']} = {nutrition_result['calories']} kcal, "
          f"{nutrition_result['protein']} protein — ideal post-run fuel.\n"
          f"  Week 1 plan: {workout_plan[:100]}...'\n")

    print('── Protocol breakdown ──────────────────────────────────────────')
    print('  MCP : Diet Coach → Nutrition DB  (tool layer — no agency)')
    print('  A2A : Diet Coach → Workout Agent (agent layer — reasons + uses MCP)')
    print('  Key : Workout Agent MCP is invisible to Diet Coach')
    print('  Result: both outputs synthesised in one Diet Coach response')


await demonstrate_mcp_and_a2a_composition()


## 6.7 · Production Patterns

### Circuit Breaker
Treats an unresponsive downstream agent the way a service mesh treats an unhealthy pod.
After `threshold` consecutive failures the circuit **opens** — requests stop immediately.
After `cooldown_s` seconds a single probe is sent. If it succeeds the circuit **closes**.

### Event-Driven vs Synchronous
Every A2A call in this chapter is **synchronous** — the caller blocks until the agent responds.
For workflows spanning hours or days, use an **event-driven** pattern instead:
- Emit task to a durable stream (Redis, SQS) and return immediately
- Downstream agent processes when available
- Result arrives as a separate event on a reply stream

In [ ]:
import time, json, os


# ── Circuit Breaker ────────────────────────────────────────────────────────
class CircuitBreaker:
    """Section 6.7.3: Resilience pattern for A2A agent calls."""
    def __init__(self, threshold=3, cooldown_s=60):
        self.failures   = 0
        self.threshold  = threshold
        self.open_until = 0.0

    def is_open(self) -> bool:
        return self.failures >= self.threshold and time.time() < self.open_until

    def record_failure(self):
        self.failures  += 1
        self.open_until = time.time() + 60

    def record_success(self):
        self.failures = 0


# Demo — circuit breaker states
cb = CircuitBreaker(threshold=3, cooldown_s=60)
print('Circuit breaker demo:')
print(f'  Initial state: open={cb.is_open()}')
for i in range(3): cb.record_failure()
print(f'  After 3 failures: open={cb.is_open()}')
cb.record_success()
print(f'  After success: open={cb.is_open()}')


# ── Event-Driven vs Synchronous ────────────────────────────────────────────
print('\n── Sync vs Event-Driven Pattern ──────────────────────────────────')
print('\nPattern 1: Synchronous A2A (used throughout this chapter)')
print('  result = await a2a_tool.ask("Create a running plan")')
print('  # Blocks until response. Suitable for sub-second tasks.')

print('\nPattern 2: Event-driven (Redis Streams — for long-running workflows)')
task_payload = {'task_id': 'task-001', 'task_type': 'create_running_plan',
                'request': '4-week 5k plan', 'callback': 'agent:results'}

redis_url = os.getenv('REDIS_URL', '')
if redis_url:
    import redis.asyncio as aioredis
    async def emit_task():
        r = await aioredis.from_url(redis_url)
        msg_id = await r.xadd('agent:tasks', {'payload': json.dumps(task_payload)})
        print(f'  Task emitted: {msg_id}')
        print(f'  Result will appear on agent:results stream.')
        await r.aclose()
    await emit_task()
else:
    print('  [Simulated] Would emit to Redis stream agent:tasks:')
    print(f'  {json.dumps(task_payload, indent=4)}')
    print('  Set REDIS_URL env var to emit to a real stream.')

print('\n── When to use each ──────────────────────────────────────────────')
print('  Synchronous : task < 30s; simple request-response')
print('  Event-driven: task spans hours/days; human approval step mid-workflow')


## 6.7 · Chapter Summary

| Concept | Section | What we built |
|---------|---------|---------------|
| **HITL Gate** | 6.2 | `HITLGate` — review queue, graduated autonomy, behavioural telemetry log |
| **ADK Workout Agent** | 6.3.1 | Google ADK agent published as an A2A-compliant server |
| **AgentCard** | 6.3.2 | A2A capability descriptor (name, URL, skills) for discovery |
| **A2A Server** | 6.3.3 | Starlette app wrapping ADK runner, executor, and task store |
| **CrewA2ATool** | 6.3.4 | A2A client with ConnectionError + TimeoutError + Exception handling |
| **LLM Council** | 6.5 | Three-stage deliberation: independent → anonymous review → synthesis |
| **MCP tool layer** | 6.6 | Workout Agent uses McpToolset internally — invisible to A2A callers |
| **Two-layer composition** | 6.6 | Diet Coach uses MCP (nutrition) + A2A (Workout Agent) simultaneously |
| **Circuit Breaker** | 6.7.3 | Opens after 3 failures; probes after cooldown; resets on success |
| **Event-driven pattern** | 6.7.6 | Redis xadd vs blocking A2A call — when to use each |

**The two-layer principle (never forget this):**
- **MCP** = tool layer → systems that respond but do not reason
- **A2A** = agent layer → peers that reason, plan, have their own tools
- Every production agent uses both. Design them as separate concerns from the start.

---
*Mastering Agentic AI · Manning Publications*